In [ ]:
# =====================================================
# ADVANCED DRIVER REALLOCATION ENGINE
# (Geospatial AI + ML + Real-time Analytics)
# =====================================================

!pip install mlflow
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import math
import folium
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import joblib
import os

try:
    import geopandas as gpd
    from shapely.geometry import Point, Polygon
    import networkx as nx
    from geopy.distance import great_circle
    from sklearn.cluster import DBSCAN
    import osmnx as ox
    print("Advanced geospatial libraries loaded successfully")
except ImportError as e:
    print(f"Some advanced libraries not installed: {e}")
    print("Core functionality will still work")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.8 MB/s eta 0:00:00
Some advanced libraries not installed: No module named 'osmnx'
Core functionality will still work


In [ ]:
# =====================================================
# 1. ZONE COORDINATES + TIME WINDOWS
# =====================================================

zone_coords = {
    "Adugodi": (12.9716, 77.5946), "Agaram": (12.8431, 77.4863),
    "Air Force Stn. Yelahanka": (13.1048, 77.5763), "Banashankari": (12.925453, 77.546761),
    "Banashankari 2nd Stage": (12.9249, 77.5662), "Banashankari 3rd Stage": (12.9271, 77.5548),
    "Bangalore City": (12.972442, 77.580643), "Bangalore GPO": (12.972442, 77.580643),
    "Bangalore University": (12.9462, 77.5103), "Bannerghatta": (12.9426, 77.6027),
    "Bannerghatta Road": (12.9426, 77.6027), "Banaswadi": (13.0108, 77.6493),
    "Basavangudi": (12.9422, 77.5748), "Basaveswaranagar": (12.9957, 77.5419),
    "Benson Town": (12.9966, 77.6042), "Byatarayanapura": (13.0588, 77.59385),
    "Carmelram": (12.9062, 77.7066), "Chamrajpet West": (12.9586, 77.5634),
    "Chickpet": (12.9708, 77.5806), "Chikkabanavara": (13.0823, 77.5068),
    "C V Raman Nagar": (12.9846, 77.6622), "Dasarahalli": (13.0458, 77.5111),
    "Dharmaram College": (12.9376, 77.5991), "Doddakallasandra": (12.8807, 77.5576),
    "Domlur": (12.9610, 77.6387), "Dooravaninagar": (13.0077, 77.6737),
    "Fraser Town": (13.0007, 77.6165), "Gandhinagar": (12.9791, 77.5777),
    "Gavipuram Extension": (12.9463, 77.5669), "GKVK": (13.0821, 77.5762),
    "Gokula Extension": (12.9716, 77.5946), "Hebbal Agri Farm": (13.0324, 77.5992),
    "Hesaraghatta": (13.1585, 77.4888), "Hesaraghatta Lake": (13.1585, 77.4888),
    "HKP Road": (12.98551, 77.60678), "HMT": (13.0311, 77.5569),
    "Hospital Town East": (12.9956, 77.6113), "Hospital Town West": (12.9716, 77.5946),
    "Indiranagar": (12.9719, 77.6412), "JC Nagar": (12.9876, 77.6379),
    "Jalahalli": (13.0519, 77.5416), "Jayanagar 3rd Block": (12.9329, 77.5839),
    "Jayanagar East": (12.9301, 77.5877), "Jayanagar South": (12.9301, 77.5877),
    "JP Nagar": (12.9105, 77.5857), "Kadugodi": (12.9986, 77.7631),
    "Kengeri": (12.9000, 77.4833), "Kothanur": (13.0585, 77.6407),
    "Krishnarajapuram": (13.0006, 77.6746), "Kumbalgodu": (12.9716, 77.5946),
    "Koramangala": (12.9317, 77.6227), "Koramangala 6th Block": (12.9382, 77.6228),
    "Madivala": (12.9211, 77.6134), "Magadi Road": (12.9709, 77.5658),
    "Mahadevapura": (12.9904, 77.6842), "Mahalakshmi Layout": (13.0114, 77.5467),
    "Malleswaram": (13.0081, 77.5648), "Malleswaram West": (12.9996, 77.5689),
    "Marathahalli Colony": (12.9512, 77.6998), "Maruthisevanagar": (13.0002, 77.6336),
    "Mathikere": (13.0320, 77.5605), "Nagarbhavi": (12.9717, 77.5132),
    "Nagasandra": (13.0289, 77.4423), "Nagashettyhalli": (12.9642, 77.6207),
    "Nandhini Layout": (13.0124, 77.5361), "Nayandahalli": (12.9396, 77.5204),
    "New Tippasandra": (12.9769, 77.6493), "Peenya": (13.0085, 77.4996),
    "Rajajinagar": (12.9906, 77.5533), "Richmond Town": (12.9634, 77.6035),
    "RT Nagar": (13.0223, 77.5949), "Seshadripuram": (12.9864, 77.5820),
    "Shanthinagar": (12.9611, 77.6047), "Srirampuram": (12.9870, 77.5662),
    "St Johns": (12.991388, 77.61186), "St Thomas Town": (13.0059, 77.6231),
    "Subramanyapura": (12.9052, 77.5433), "Thyagarajanagar": (12.9293, 77.5680),
    "Ulsoor": (12.9815, 77.6192), "Vasanthanagar": (12.9911, 77.5920),
    "Vidyaranyapura": (13.0754, 77.5591), "Vijayanagar": (12.9699, 77.5333),
    "Vimanapura": (12.9645, 77.6865), "Virgonagar": (13.0319, 77.7322),
    "Vyalikaval": (13.0041, 77.5749), "Viveknagar": (12.9496, 77.6223),
    "Whitefield": (12.9698, 77.7499), "Wilson Garden": (12.9490, 77.5978),
    "Yelahanka": (13.1048, 77.5763), "Yeswanthpur": (13.0178, 77.5572),
    "BTM Layout": (12.9166, 77.6101), "HSR Layout": (12.9081, 77.6476),
    "Electronic City": (12.8452, 77.6600), "Sadashivanagar": (13.0102, 77.5770),
    "Bellandur": (12.9304, 77.6784), "Sarjapur Road": (12.9121, 77.6865),
    "Yeshwanthpur Industrial Area": (13.0205, 77.5412), "Basavanagudi": (12.9416, 77.5713),
    "Attibele": (12.7785, 77.7706), "Anekal": (12.7120, 77.6954),
    "Bommasandra": (12.8213, 77.7056), "Chandapura": (12.8001, 77.7052),
    "Rajarajeshwari Nagar": (12.9155, 77.5204), "Padmanabhanagar": (12.9175, 77.5581),
    "Uttarahalli": (12.9060, 77.5528), "Banerghatta Main": (12.8993, 77.5965),
    "Hebbal Kempapura": (13.0352, 77.5912), "RMV Extension": (13.0316, 77.5694),
    "Yemalur": (12.9534, 77.6722), "Nagavara": (13.0450, 77.6240),
    "Hoodi": (12.9920, 77.7150), "Brookefield": (12.9672, 77.7180),
    "Horamavu": (13.0215, 77.6408), "Ramamurthy Nagar": (13.0125, 77.6773),
    "Kalyan Nagar": (13.0234, 77.6402), "Kammanahalli": (13.0207, 77.6419),
    "Yelahanka New Town": (13.1000, 77.5800), "Majestic": (12.9767, 77.5713),
    "Cottonpet": (12.9690, 77.5654), "Chamarajpet": (12.9568, 77.5650),
    "Ulsoor Lake": (12.9825, 77.6215), "Ejipura": (12.9384, 77.6258),
    "Ashok Nagar": (12.9702, 77.6019), "Frazer Town East": (13.0024, 77.6189),
    "Lingarajapuram": (13.0138, 77.6306), "Nagadevanahalli": (12.9720, 77.4742),
    "Anjanapura": (12.8820, 77.5602), "Haralur": (12.9012, 77.6664),
    "Kasavanahalli": (12.9083, 77.6655), "Chikkalasandra": (12.9270, 77.5460),
    "Girinagar": (12.9450, 77.5532), "Kathriguppe": (12.9245, 77.5488),
    "Basaveshwar Nagar": (12.9960, 77.5410),
}

zones = list(zone_coords.keys())
print(f"Loaded {len(zones)} zones successfully")

time_windows = [
    "00:00", "01:00", "02:00", "03:00", "04:00", "05:00",
    "06:00", "07:00", "08:00", "09:00", "10:00", "11:00",
    "12:00", "13:00", "14:00", "15:00", "16:00", "17:00",
    "18:00", "19:00", "20:00", "21:00", "22:00", "23:00",
]


Loaded 133 zones successfully


In [ ]:
# =====================================================
# 2. HAVERSINE DISTANCE FUNCTION
# =====================================================

def haversine(c1, c2):
    lat1, lon1 = c1
    lat2, lon2 = c2
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2 +
         math.cos(math.radians(lat1)) *
         math.cos(math.radians(lat2)) *
         math.sin(dlon/2)**2)
    return 2 * R * math.asin(math.sqrt(a))

In [ ]:
# =====================================================
# 3. GENERATE SYNTHETIC DEMAND/SUPPLY DATA
# =====================================================

np.random.seed(42)
data_rows = []

def get_demand_multiplier(time_str):
    hour = int(time_str.split(":")[0])
    if 0 <= hour < 5:    return 0.3
    elif 5 <= hour < 7:  return 0.6
    elif 7 <= hour < 9:  return 1.8
    elif 9 <= hour < 12: return 1.2
    elif 12 <= hour < 14: return 1.5
    elif 14 <= hour < 17: return 1.1
    elif 17 <= hour < 20: return 2.2
    elif 20 <= hour < 22: return 1.6
    else:                return 0.8

zone_type_multipliers = {
    "Whitefield": 1.5, "Electronic City": 1.5, "ITPL": 1.5,
    "Marathahalli Colony": 1.4, "Indiranagar": 1.4, "Koramangala": 1.4,
    "HSR Layout": 1.3, "BTM Layout": 1.3, "Bellandur": 1.4,
    "Sarjapur Road": 1.3, "Majestic": 1.4, "Yeswanthpur": 1.3,
    "Malleswaram": 1.2, "Jayanagar 3rd Block": 1.2, "JP Nagar": 1.2,
    "Rajajinagar": 1.1, "Yelahanka": 1.0, "Hebbal": 1.1,
}

for t in time_windows:
    time_mult = get_demand_multiplier(t)
    for z in zones:
        zone_mult = zone_type_multipliers.get(z, 1.0)
        base_demand = np.random.poisson(10) * time_mult * zone_mult
        base_supply = np.random.poisson(8) * time_mult * zone_mult * 0.9
        demand = max(0, int(np.random.normal(base_demand, base_demand * 0.2)))
        supply = max(0, int(np.random.normal(base_supply, base_supply * 0.15)))
        data_rows.append({"Time_Window": t, "Zone": z, "Demand": demand, "Supply": supply})

df_zone_status = pd.DataFrame(data_rows)
df_zone_status["Gap"] = df_zone_status["Demand"] - df_zone_status["Supply"]


In [ ]:
# =====================================================
# 4. BUILD TRAINING DATASET
# =====================================================

training_rows = []
for _, row in df_zone_status.iterrows():
    current_zone   = row["Zone"]
    current_time   = row["Time_Window"]
    current_supply = row["Supply"]
    current_demand = row["Demand"]
    current_gap    = row["Gap"]

    hour    = int(current_time.split(":")[0])
    is_peak = (7 <= hour <= 9) or (17 <= hour <= 20)

    max_search_radius = 3.0 if (current_gap > 0 or is_peak) else 4.0

    distances = []
    for z in zones:
        if z == current_zone: continue
        d = haversine(zone_coords[current_zone], zone_coords[z])
        distances.append((z, d))
    distances.sort(key=lambda x: x[1])
    nearest_zones = [z for z, _ in distances[:2]]

    gap1 = df_zone_status[(df_zone_status.Zone == nearest_zones[0]) & (df_zone_status.Time_Window == current_time)].Gap.values[0] if len(nearest_zones) > 0 else 0
    gap2 = df_zone_status[(df_zone_status.Zone == nearest_zones[1]) & (df_zone_status.Time_Window == current_time)].Gap.values[0] if len(nearest_zones) > 1 else 0

    best_score = -np.inf
    best_zone  = current_zone

    for z in zones:
        if z == current_zone: continue
        dist = haversine(zone_coords[current_zone], zone_coords[z])
        if dist <= max_search_radius:
            zg = df_zone_status[(df_zone_status.Zone == z) & (df_zone_status.Time_Window == current_time)].Gap.values[0]
            if zg > 0:
                score = zg / (dist + 1)
                if score > best_score:
                    best_score = score
                    best_zone  = z

    if current_gap > 0 and best_zone != current_zone:
        if (current_gap / 1.0) >= best_score:
            best_zone = current_zone

    training_rows.append({
        "Time_Window": current_time, "Current_Zone": current_zone,
        "Current_Zone_Supply": current_supply, "Current_Zone_Demand": current_demand,
        "Surrounding_Zone_1_Gap": gap1, "Surrounding_Zone_2_Gap": gap2,
        "Target_Variable": best_zone,
    })

df_train = pd.DataFrame(training_rows)

In [ ]:
# =====================================================
# 5. ENCODING
# =====================================================

le_time        = LabelEncoder()
le_input_zone  = LabelEncoder()
le_target_zone = LabelEncoder()

df_train["Time_Window_enc"]  = le_time.fit_transform(df_train["Time_Window"])
df_train["Current_Zone_enc"] = le_input_zone.fit_transform(df_train["Current_Zone"])
df_train["Target_enc"]       = le_target_zone.fit_transform(df_train["Target_Variable"])

In [ ]:
# =====================================================
# 6. ENHANCED FEATURE ENGINEERING
# =====================================================

def create_enhanced_features(df):
    df = df.copy()
    df['Hour']                  = df['Time_Window'].apply(lambda x: int(x.split(':')[0]))
    df['Hour_Sin']              = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_Cos']              = np.cos(2 * np.pi * df['Hour'] / 24)
    df['Supply_Demand_Ratio']   = df['Current_Zone_Supply'] / (df['Current_Zone_Demand'] + 1)
    df['Log_Supply']            = np.log1p(df['Current_Zone_Supply'])
    df['Log_Demand']            = np.log1p(df['Current_Zone_Demand'])
    df['Abs_Gap']               = np.abs(df['Current_Zone_Demand'] - df['Current_Zone_Supply'])
    df['Is_Deficit']            = (df['Current_Zone_Demand'] > df['Current_Zone_Supply']).astype(int)
    df['Is_Surplus']            = (df['Current_Zone_Demand'] < df['Current_Zone_Supply']).astype(int)
    df['Avg_Surrounding_Gap']   = (df['Surrounding_Zone_1_Gap'] + df['Surrounding_Zone_2_Gap']) / 2
    df['Max_Surrounding_Gap']   = df[['Surrounding_Zone_1_Gap', 'Surrounding_Zone_2_Gap']].max(axis=1)
    df['Total_Surrounding_Gap'] = df['Surrounding_Zone_1_Gap'] + df['Surrounding_Zone_2_Gap']
    return df

df_train = create_enhanced_features(df_train)

feature_columns = [
    'Time_Window_enc', 'Current_Zone_enc', 'Current_Zone_Supply', 'Current_Zone_Demand',
    'Surrounding_Zone_1_Gap', 'Surrounding_Zone_2_Gap', 'Hour_Sin', 'Hour_Cos',
    'Supply_Demand_Ratio', 'Log_Supply', 'Log_Demand', 'Abs_Gap',
    'Is_Deficit', 'Is_Surplus', 'Avg_Surrounding_Gap', 'Max_Surrounding_Gap', 'Total_Surrounding_Gap',
]

X = df_train[feature_columns]
y = df_train['Target_enc']


In [ ]:
# =====================================================
# 7. TRAIN MODEL  +  LOG WITH MLFLOW
# =====================================================

print("\n" + "=" * 50)
print("TRAINING MODEL")
print("=" * 50)

mlflow.set_experiment("driver_reallocation")           # ← creates experiment in Databricks

with mlflow.start_run(run_name="RandomForest_v1"):

    # ── Grid Search ──────────────────────────────────
    param_grid = {
        'n_estimators':      [200, 300],
        'max_depth':         [10, 15, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf':  [1, 2],
    }

    rf = RandomForestClassifier(random_state=42)
    grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
    grid_search.fit(X, y)

    print(f"Best parameters : {grid_search.best_params_}")
    print(f"Best CV score   : {grid_search.best_score_:.4f}")

    # ── Train final model ────────────────────────────
    best_model = RandomForestClassifier(**grid_search.best_params_, random_state=42)
    best_model.fit(X, y)

    train_acc = accuracy_score(y, best_model.predict(X))

    # ── Log to MLflow ────────────────────────────────
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("cv_accuracy",    round(grid_search.best_score_, 4))
    mlflow.log_metric("train_accuracy", round(train_acc, 4))
    mlflow.sklearn.log_model(best_model, artifact_path="model")

    print(f"\nCV Accuracy    : {grid_search.best_score_:.4f}")
    print(f"Train Accuracy : {train_acc:.4f}")
    print(f"Run ID         : {mlflow.active_run().info.run_id}")
    print("Model saved to MLflow ✅")

# ── Feature importance ───────────────────────────────
feature_importance = pd.DataFrame({
    'feature':    feature_columns,
    'importance': best_model.feature_importances_,
}).sort_values('importance', ascending=False)

print("\nTop 10 Features:")
print(feature_importance.head(10))


TRAINING MODEL


2026/03/18 03:45:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/18 03:45:48 INFO mlflow.store.db.utils: Updating database tables
2026/03/18 03:45:54 INFO mlflow.tracking.fluent: Experiment with name 'driver_reallocation' does not exist. Creating a new experiment.


Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best parameters : {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Best CV score   : 0.0695


2026/03/18 03:49:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/18 03:49:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



CV Accuracy    : 0.0695
Train Accuracy : 0.6729
Run ID         : 4a1e83ed51b5425ebdc304a4579a4cfd
Model saved to MLflow ✅

Top 10 Features:
                   feature  importance
1         Current_Zone_enc    0.240783
5   Surrounding_Zone_2_Gap    0.067359
15     Max_Surrounding_Gap    0.067108
4   Surrounding_Zone_1_Gap    0.064052
8      Supply_Demand_Ratio    0.064035
14     Avg_Surrounding_Gap    0.060617
16   Total_Surrounding_Gap    0.058389
7                 Hour_Cos    0.047983
0          Time_Window_enc    0.047531
3      Current_Zone_Demand    0.046529


In [ ]:
# =====================================================
# 8. RECOMMENDATION FUNCTION
# =====================================================

def get_gaps_for_time(time_window):
    return df_zone_status[df_zone_status['Time_Window'] == time_window].set_index('Zone')['Gap'].to_dict()

def advanced_recommendation(input_dict):
    time         = input_dict["Time_Window"]
    current_zone = input_dict["Current_Zone"]
    supply       = input_dict["Current_Zone_Supply"]
    demand       = input_dict["Current_Zone_Demand"]
    current_gap  = demand - supply

    hour    = int(time.split(':')[0])
    is_peak = (7 <= hour <= 9) or (17 <= hour <= 20)
    max_distance_km = 3.0 if (current_gap > 0 or is_peak) else 4.0

    gaps = get_gaps_for_time(time)

    distances_for_input = sorted(
        [(z, haversine(zone_coords[current_zone], zone_coords[z])) for z in zones if z != current_zone],
        key=lambda x: x[1]
    )
    nearest_zones = [z for z, _ in distances_for_input[:2]]

    gap1 = gaps.get(nearest_zones[0], 0) if nearest_zones else 0
    gap2 = gaps.get(nearest_zones[1], 0) if len(nearest_zones) > 1 else 0

    input_ml_df = pd.DataFrame({
        "Time_Window": [time], "Current_Zone": [current_zone],
        "Current_Zone_Supply": [supply], "Current_Zone_Demand": [demand],
        "Surrounding_Zone_1_Gap": [gap1], "Surrounding_Zone_2_Gap": [gap2],
    })

    input_ml_df["Time_Window_enc"]  = le_time.transform(input_ml_df["Time_Window"])
    input_ml_df["Current_Zone_enc"] = (
        le_input_zone.transform(input_ml_df["Current_Zone"])
        if current_zone in le_input_zone.classes_ else [0]
    )
    input_ml_df = create_enhanced_features(input_ml_df)
    input_ml    = input_ml_df[feature_columns]

    ml_pred_enc = best_model.predict(input_ml)[0]
    ml_zone     = le_target_zone.inverse_transform([ml_pred_enc])[0]

    scores = []
    for zone in zones:
        if zone == current_zone: continue
        dist = haversine(zone_coords[current_zone], zone_coords[zone])
        if dist <= max_distance_km:
            gap = gaps.get(zone, 0)
            if gap > 0:
                time_factor  = 1.5 if is_peak else 1.0
                zone_mult    = zone_type_multipliers.get(zone, 1.0)
                dist_penalty = (dist ** 1.5) if is_peak else dist
                score = (gap * zone_mult * time_factor) / (dist_penalty + 1)
                scores.append({'zone': zone, 'distance': round(dist, 2), 'gap': gap, 'score': round(score, 2)})

    scores.sort(key=lambda x: x['score'], reverse=True)
    ml_dist = haversine(zone_coords[current_zone], zone_coords[ml_zone]) if ml_zone != current_zone else 0

    if current_gap > 0:
        if ml_zone == current_zone:
            best_zone, reason = current_zone, "Local Deficit: ML recommends staying in current location."
        elif ml_dist <= max_distance_km and gaps.get(ml_zone, 0) > 0:
            best_zone, reason = ml_zone, "Local Deficit: ML suggests moving to a nearby zone with higher deficit."
        elif scores and scores[0]['zone'] != current_zone and scores[0]['distance'] <= max_distance_km and scores[0]['gap'] > current_gap:
            best_zone, reason = scores[0]['zone'], "Local Deficit: Rule-based suggests moving to a nearby zone with higher deficit."
        else:
            best_zone, reason = current_zone, "Local Deficit: Driver needed in current location. Stay."
    else:
        if ml_zone != current_zone and ml_dist <= max_distance_km and gaps.get(ml_zone, 0) > 0:
            best_zone, reason = ml_zone, "Surplus re-routing: Utilizing ML optimal target."
        elif scores and scores[0]['zone'] != current_zone and scores[0]['distance'] <= max_distance_km and scores[0]['gap'] > 0:
            best_zone, reason = scores[0]['zone'], "Surplus re-routing: ML overridden or recommended staying. Using best rule-based local score."
        else:
            best_zone, reason = current_zone, "Surplus, but no high-demand zones within allowable radius. STAY."

    return {
        'ml_zone': ml_zone, 'recommended_zone': best_zone, 'reason': reason,
        'rankings': scores[:5],
        'current_zone_info': {
            'zone': current_zone, 'supply': supply, 'demand': demand,
            'gap': current_gap, 'max_allowed_radius': max_distance_km,
        },
    }

In [ ]:
# =====================================================
# 9. TEST WITH SAMPLE DRIVER
# =====================================================

sample_driver = {
    "Time_Window":         "14:00",
    "Current_Zone":        "Basaveswaranagar",
    "Current_Zone_Supply": 40,
    "Current_Zone_Demand": 23,
}

print("\n" + "=" * 50)
print("SAMPLE RECOMMENDATION")
print("=" * 50)
print(f"Driver Location : {sample_driver['Current_Zone']}")
print(f"Time            : {sample_driver['Time_Window']}")
print(f"Supply          : {sample_driver['Current_Zone_Supply']}")
print(f"Demand          : {sample_driver['Current_Zone_Demand']}")

result = advanced_recommendation(sample_driver)

print(f"\nML Suggested Zone  : {result['ml_zone']}")
print(f"Final Recommendation: {result['recommended_zone']}")
print(f"Reason             : {result['reason']}")
print("\nTop 5 Recommended Zones:")
for i, r in enumerate(result['rankings'][:5], 1):
    print(f"  {i}. {r['zone']} — Score: {r['score']}, Distance: {r['distance']} km, Gap: {r['gap']}")


SAMPLE RECOMMENDATION
Driver Location : Basaveswaranagar
Time            : 14:00
Supply          : 40
Demand          : 23

ML Suggested Zone  : Electronic City
Final Recommendation: Nandhini Layout
Reason             : Surplus re-routing: ML overridden or recommended staying. Using best rule-based local score.

Top 5 Recommended Zones:
  1. Nandhini Layout — Score: 2.7, Distance: 1.96 km, Gap: 8
  2. Rajajinagar — Score: 1.87, Distance: 1.36 km, Gap: 4
  3. Malleswaram West — Score: 1.77, Distance: 2.96 km, Gap: 7
  4. Yeshwanthpur Industrial Area — Score: 1.6, Distance: 2.76 km, Gap: 6
  5. Majestic — Score: 1.45, Distance: 3.82 km, Gap: 5


In [ ]:
# =====================================================
# 10. INTERACTIVE MAP — USER INPUT + RECOMMENDATION
# =====================================================

import folium
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ── Custom CSS ───────────────────────────────────────────────
custom_css = HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Sora:wght@300;400;600;700&display=swap');
  :root {
    --bg:#0d1117; --surface:#161b22; --border:#30363d;
    --accent:#00d4aa; --accent2:#f78166; --accent3:#79c0ff;
    --text:#e6edf3; --muted:#8b949e; --success:#3fb950;
  }
  .panel {
    background:var(--bg); border:1px solid var(--border);
    border-radius:16px; padding:24px 28px;
    font-family:'Sora',sans-serif; color:var(--text);
    max-width:760px; margin:12px 0;
    box-shadow:0 0 40px rgba(0,212,170,.08);
  }
  .panel-title {
    font-size:13px; font-weight:700; letter-spacing:.1em;
    text-transform:uppercase; color:var(--accent);
    margin-bottom:20px; padding-bottom:12px;
    border-bottom:1px solid var(--border);
    display:flex; align-items:center; gap:10px;
  }
  .dot { width:8px; height:8px; border-radius:50%; background:var(--accent); box-shadow:0 0 6px var(--accent); display:inline-block; }
  .hero-grid { display:grid; grid-template-columns:1fr 1fr; gap:14px; margin-bottom:20px; }
  .hero-card { background:var(--surface); border:1px solid var(--border); border-radius:12px; padding:18px; }
  .hero-card .lbl { font-size:9px; letter-spacing:.12em; text-transform:uppercase; color:var(--muted); margin-bottom:6px; }
  .hero-card .val { font-size:20px; font-weight:700; color:var(--accent); }
  .hero-card .sub { font-size:10px; color:var(--muted); margin-top:3px; font-family:'JetBrains Mono',monospace; }
  .hero-card.ml .val { color:var(--accent3); }
  .hero-card.ml { border-color:rgba(121,192,255,.2); }
  .reason-box {
    background:rgba(0,212,170,.06); border:1px solid rgba(0,212,170,.2);
    border-radius:10px; padding:12px 16px; font-size:12px; color:var(--accent);
    margin-bottom:20px; display:flex; gap:10px;
  }
  .rankings-title { font-size:10px; letter-spacing:.1em; text-transform:uppercase; color:var(--muted); margin-bottom:10px; }
  .rank-row {
    display:grid; grid-template-columns:24px 1fr 80px 80px 70px;
    align-items:center; gap:10px; padding:10px 12px;
    border-radius:8px; margin-bottom:5px;
    background:var(--surface); border:1px solid var(--border); font-size:12px;
  }
  .rank-row.top { background:rgba(0,212,170,.08); border-color:rgba(0,212,170,.4); }
  .rank-num { font-family:'JetBrains Mono',monospace; font-size:10px; color:var(--muted); text-align:center; }
  .rank-row.top .rank-num { color:var(--accent); font-weight:700; }
  .rank-zone { font-weight:600; }
  .badge { font-family:'JetBrains Mono',monospace; font-size:10px; padding:2px 7px; border-radius:4px; text-align:center; }
  .bs { background:rgba(63,185,80,.15); color:var(--success); }
  .bd { background:rgba(121,192,255,.12); color:var(--accent3); }
  .bg { background:rgba(247,129,102,.12); color:var(--accent2); }
</style>
""")


# ── Result card renderer ─────────────────────────────────────
def render_result_html(result, driver):
    import datetime
    ts = datetime.datetime.now().strftime("%H:%M:%S")

    rows = ""
    for i, r in enumerate(result["rankings"][:5], 1):
        cls = "top" if i == 1 else ""
        rows += f"""
        <div class="rank-row {cls}">
          <span class="rank-num">#{i}</span>
          <span class="rank-zone">{r['zone']}</span>
          <span class="badge bs">Score {r['score']}</span>
          <span class="badge bd">{r['distance']} km</span>
          <span class="badge bg">Gap {r['gap']}</span>
        </div>"""

    return HTML(f"""
    <div class="panel">
      <div class="panel-title"><span class="dot"></span> Relocation Result &nbsp;·&nbsp; {ts}</div>
      <div class="hero-grid">
        <div class="hero-card">
          <div class="lbl">Final Recommendation</div>
          <div class="val">{result['recommended_zone']}</div>
          <div class="sub">Rule-constrained best zone</div>
        </div>
        <div class="hero-card ml">
          <div class="lbl">ML Suggested Zone</div>
          <div class="val">{result['ml_zone']}</div>
          <div class="sub">Raw model prediction</div>
        </div>
      </div>
      <div class="reason-box">💡 &nbsp;{result['reason']}</div>
      <div class="rankings-title">Top 5 Candidate Zones</div>
      {rows}
    </div>""")


# ── Map builder ──────────────────────────────────────────────
def build_map(driver, result):
    current_zone     = driver["Current_Zone"]
    recommended_zone = result["recommended_zone"]
    ml_zone          = result["ml_zone"]
    rankings         = result["rankings"]
    max_radius_km    = result["current_zone_info"]["max_allowed_radius"]

    start = zone_coords[current_zone]
    end   = zone_coords[recommended_zone]

    m = folium.Map(location=start, zoom_start=13, tiles="CartoDB dark_matter")

    # Radius circle
    folium.Circle(
        location=start, radius=max_radius_km * 1000,
        color="#79c0ff", fill=False, weight=1.5, dash_array="6",
        tooltip=f"Max radius: {max_radius_km} km",
    ).add_to(m)

    # Route line
    if current_zone != recommended_zone:
        folium.PolyLine(
            [start, end], weight=3, color="#00d4aa",
            tooltip=f"Route to {recommended_zone}",
        ).add_to(m)

    # Top 5 candidates
    for i, r in enumerate(rankings[:5], 1):
        folium.CircleMarker(
            location=zone_coords[r["zone"]], radius=8,
            color="#f78166", fill=True, fill_color="#f78166", fill_opacity=0.5,
            tooltip=f"#{i} {r['zone']} | Score:{r['score']} | Gap:{r['gap']} | {r['distance']}km",
        ).add_to(m)

    # ML zone
    if ml_zone != current_zone and ml_zone != recommended_zone:
        folium.CircleMarker(
            location=zone_coords[ml_zone], radius=10,
            color="#79c0ff", fill=True, fill_color="#79c0ff", fill_opacity=0.5,
            tooltip=f"ML Suggested: {ml_zone}",
        ).add_to(m)

    # Driver pin
    folium.Marker(
        location=start,
        tooltip=f"🚖 Driver: {current_zone}",
        popup=folium.Popup(
            f"<b>Driver Location</b><br>{current_zone}<br>"
            f"Supply: {driver['Current_Zone_Supply']}<br>"
            f"Demand: {driver['Current_Zone_Demand']}<br>"
            f"Gap: {result['current_zone_info']['gap']}",
            max_width=200
        ),
        icon=folium.Icon(color="blue", icon="car", prefix="fa"),
    ).add_to(m)

    # Recommended pin
    folium.Marker(
        location=end,
        tooltip=f"✅ Go to: {recommended_zone}",
        popup=folium.Popup(
            f"<b>Recommended Zone</b><br>{recommended_zone}<br>{result['reason']}",
            max_width=250
        ),
        icon=folium.Icon(color="green", icon="flag", prefix="fa"),
    ).add_to(m)

    # Legend
    m.get_root().html.add_child(folium.Element("""
    <div style="position:fixed;top:16px;right:16px;z-index:1000;
        background:#0d1117;border:1px solid #30363d;border-radius:10px;
        padding:14px 18px;font-family:monospace;font-size:12px;color:#e6edf3;
        box-shadow:0 0 20px rgba(0,0,0,0.5);">
        <b style="font-size:13px;color:#00d4aa;">Map Legend</b><br><br>
        🔵 Driver Location<br>🟢 Recommended Zone<br>
        <span style="color:#00d4aa">━━</span> Route<br>
        🟠 Candidate Zones<br>🔵 ML Suggested Zone<br>
        <span style="color:#79c0ff">- - -</span> Allowed Radius
    </div>"""))

    return m


# ── Widgets ──────────────────────────────────────────────────
style  = {"description_width": "160px"}
layout = widgets.Layout(width="480px")

time_dd = widgets.Dropdown(
    options     = [f"{i:02d}:00" for i in range(24)],
    value       = "14:00",
    description = "⏰  Time Window",
    style=style, layout=layout,
)
zone_dd = widgets.Dropdown(
    options     = zones,
    value       = "Basaveswaranagar",
    description = "📍  Current Zone",
    style=style, layout=layout,
)
supply_w = widgets.IntSlider(
    value=10, min=1, max=100, step=1,
    description = "🚖  Drivers Available",
    style=style, layout=layout,
    continuous_update=False,
)
demand_w = widgets.IntSlider(
    value=12, min=1, max=100, step=1,
    description = "📲  Ride Requests",
    style=style, layout=layout,
    continuous_update=False,
)

# Live gap indicator
gap_label = widgets.HTML(value="<span style='font-family:monospace;color:#8b949e'>Gap: 0</span>")

def update_gap(*args):
    gap = demand_w.value - supply_w.value
    color  = "#f78166" if gap > 0 else "#3fb950"
    status = "Deficit 🔴" if gap > 0 else "Surplus 🟢"
    gap_label.value = (
        f"<span style='font-family:monospace;color:{color};font-size:13px'>"
        f"Gap: {gap:+d}  →  {status}</span>"
    )

supply_w.observe(update_gap, names="value")
demand_w.observe(update_gap, names="value")
update_gap()

run_btn = widgets.Button(
    description  = "  Get Recommendation",
    button_style = "success",
    icon         = "location-arrow",
    layout       = widgets.Layout(width="220px", height="40px", margin="16px 0 0 0"),
)

output = widgets.Output()

# ── Button click ─────────────────────────────────────────────
def on_click(b):
    with output:
        clear_output(wait=True)

        driver = {
            "Time_Window":         time_dd.value,
            "Current_Zone":        zone_dd.value,
            "Current_Zone_Supply": supply_w.value,
            "Current_Zone_Demand": demand_w.value,
        }

        result = advanced_recommendation(driver)

        display(custom_css)
        display(render_result_html(result, driver))

        try:
            display(build_map(driver, result))
        except Exception as e:
            display(HTML(f'<p style="color:#f78166;font-family:monospace">Map error: {e}</p>'))

run_btn.on_click(on_click)

# ── Layout ───────────────────────────────────────────────────
input_box = widgets.HTML("""
<div style="
    font-family:'Sora',sans-serif; background:#0d1117;
    border:1px solid #30363d; border-radius:14px;
    padding:18px 24px; max-width:520px; margin-bottom:10px;
">
  <div style="font-size:12px;font-weight:700;letter-spacing:.1em;
    text-transform:uppercase;color:#00d4aa;margin-bottom:16px;">
    🚖 Driver State Input
  </div>
</div>
""")

display(custom_css)
display(widgets.VBox([
    input_box,
    time_dd,
    zone_dd,
    supply_w,
    demand_w,
    gap_label,
    run_btn,
    output,
]))